# Vérification des balises

## Chargement des bibliothèques 

In [16]:
import csv
import xml.etree.ElementTree as ET
import os
from collections import defaultdict

## Objectifs de ce Notebook

On veut identifier les erreurs dans chaque fichier XML du corpus
- Vérification des identifiants : 
    * Comparaisons entre les identifiants de l'Index, du sheets et des refs placées dans les textes

Vérification en cours
- vérifier que pers = Personne et place = Lieux
- vérifier que la ref est associé à une des possibilité d'écriture de la liste des variantes ??

## Création des fonctions

### Extractions des persName, variantes et ids

Création des listes et de fichiers csv pour lires les différentes informations recherchées.

On passe tous les fichiers .xml du corpus ArTerm en revue afin d'extraire le texte contenu dans les balises persName pour avoir une liste des variantes utilisées dans le corpus pour chacun des identifiants. Cette liste se trouve dans le fichier `variantes.csv`. 

On obtient également une liste des noms balisés persNames qui n'ont pas été identifiés, ou pas indexés. La liste se trouve dans le fichier `Non_ID.csv`. La sortie est triée par fichier xml.

Enfin on a une liste de toutes les références utilisées dans chaque fichier xml. Cette liste se trouve dans le fichier `refParTexte.csv`.

In [12]:
def extract_texts_from_persname(folder_path, sortiePers, NaNPers, refPersInTxt):
    variantes_noms = defaultdict(set) # Création de la liste des variantes des noms
    namespace = {'tei': 'http://www.tei-c.org/ns/1.0'} # Namespace des fichiers en XML/TEI pour que le parser fonctionne
    non_identifies = defaultdict(set) # Création de la liste des noms balisés mais non identifiés
    refs_texte = defaultdict(set) # Création d'une liste qui récupère tous les identifiants trouvés dans les balises persName
    
    total_persname = 0
    
    for root_dir, dirs, files in os.walk(folder_path):  # Parcourt récursivement tous les sous-dossiers
        print("Directory path: %s" % root_dir)  # Correction: root_dir au lieu de root
        print("Directory Names: %s" % dirs)
        print("Files Names: %s" % files)
        for file_name in files:
            if not file_name.startswith('Index') and file_name.endswith('.xml'):  # Permet de lire tous les fichiers .xml en excluant les deux Index.
                file_path = os.path.join(root_dir, file_name)  # Utilise root_dir au lieu de folder_path
                print(f"Traitement du fichier : {file_name}")  # Permet de vérifier que tous les fichiers soient traités
                try:
                    # Lecture du fichier XML
                    tree = ET.parse(file_path)
                    root = tree.getroot()
                    
                    persname_count = len(root.findall('.//tei:persName', namespaces=namespace))
                    total_persname += persname_count

                    for name in root.findall('.//tei:persName', namespaces=namespace):  # Trouve toutes les balises persName
                        pers = name.text.strip() if name.text else ''  # Récupère le texte contenu à l'interieur des balises persName
                        ref = name.attrib.get('ref', '').strip()  # Récupère le texte contenu dans l'attribut @ref de chaque persName

                        if not ref:  # Si la balise ne contient pas d'attribut ref on le met dans la liste des non identifiés
                            if pers:  # Seulement si le nom n'est pas vide
                                non_identifies[file_name].add(pers)
                        else:  # Sinon on supprime le # en début d'identifiant et on ajoute l'id et le contenu entre les balises dans la liste des variantes.
                            if ref.startswith('#'):
                                ref = ref[1:]
                            if pers:  # Seulement si le nom n'est pas vide
                                variantes_noms[ref].add(pers)
                            refs_texte[file_name].add(ref)
                    
                except Exception as e:
                    print(f"Erreur dans le fichier {file_name}: {e}")
    
    sorted_var = sorted(variantes_noms.items(), key=lambda x: x[0])  # Tri des variantes par ID (ordre alphabétique)
   
    with open(sortiePers, 'w', newline='', encoding='utf-8') as csvfile:  # Ecriture d'un fichier csv avec les variantes d'écriture pour chaque indentifiant
        csvwriter = csv.writer(csvfile)
        csvwriter.writerow(['ID', 'Noms'])
        
        for ref, noms in sorted_var:
            csvwriter.writerow([ref, ','.join(sorted(noms, key=len, reverse=True))])

    with open(NaNPers, 'w', newline='', encoding='utf-8') as NonDef:  # Ecriture d'un fichier csv avec la liste des non identifiés
        inconnu = csv.writer(NonDef)
        inconnu.writerow(['fileName', 'Noms'])

        for file_name, ref in non_identifies.items():
            inconnu.writerow([file_name, ','.join(sorted(ref))])

    with open(refPersInTxt, 'w', newline='', encoding='utf-8') as partxt:  # Ecriture d'un fichier csv avec la liste complète des références dans chaque texte.
        persName = csv.writer(partxt)
        persName.writerow(['fileName', 'Ref'])
        
        for file_name, ref in refs_texte.items():
            persName.writerow([file_name, ','.join(sorted(ref))])

    print(f"Nombre total de balises persName trouvées: {total_persname}")  # Correction: persName au lieu de objectName

    return variantes_noms, non_identifies, refs_texte

Lien vers les fichiers d'entrée et de sortie de la fonction.

In [13]:
# Le chemin vers les fichiers à traiter avec la fonction. Ici vers le dépôt complet Arterm car les fichiers xml s'y trouve
# Le chemin peut être modifier si la structure du dépôt change.
folder_path = "../../../GitHubArTerm/corpus"

Liste des variables pour l'extraction des persName

In [14]:
# Noms donnés aux fichiers de sortie. 
# Ils sont dirigés vers un dossier persName.
sortiePers = "persName/variantes.csv"
NaNPers = "persName/Non_ID.csv"
refPersInTxt = 'persName/refParTexte.csv'
erreurPers = "persName/pbParTexte.csv"

Test de la fonction d'extraction, qui fonctionne seule si on veut juste sortir les fichiers de variantes, de pers non identifiés et des refs par texte.

In [15]:
extract_texts_from_persname(folder_path, sortiePers, NaNPers, refPersInTxt)

Directory path: ../../../GitHubArTerm/corpus
Directory Names: ['.git', 'Architecture', 'Peinture', 'Perspective']
Files Names: ['IndexLieux.xml', 'IndexOeuvres.xml', 'IndexPersonnes.xml', 'tei_arterm.rng']
Directory path: ../../../GitHubArTerm/corpus\.git
Directory Names: ['hooks', 'info', 'logs', 'objects', 'refs']
Files Names: ['COMMIT_EDITMSG', 'config', 'description', 'FETCH_HEAD', 'HEAD', 'index', 'ORIG_HEAD']
Directory path: ../../../GitHubArTerm/corpus\.git\hooks
Directory Names: []
Files Names: ['applypatch-msg.sample', 'commit-msg.sample', 'fsmonitor-watchman.sample', 'post-update.sample', 'pre-applypatch.sample', 'pre-commit.sample', 'pre-merge-commit.sample', 'pre-push.sample', 'pre-rebase.sample', 'pre-receive.sample', 'prepare-commit-msg.sample', 'push-to-checkout.sample', 'sendemail-validate.sample', 'update.sample']
Directory path: ../../../GitHubArTerm/corpus\.git\info
Directory Names: []
Files Names: ['exclude']
Directory path: ../../../GitHubArTerm/corpus\.git\logs
Di

(defaultdict(set,
             {'Vitruve': {'Marco Vitruvio Pollione',
               'Victruve',
               'Vitr.',
               'Vitreuve',
               'Vitruv.',
               'Vitruve',
               'Vitruve Polion',
               'Vitruvio',
               'Vittruvio',
               'cet Auteur'},
              'Archimede': {'Archimede', 'Archimedi'},
              'Jupiter': {'Giove',
               'Jupiter',
               'Jupiter Alphiste',
               'Jupiter Ammon',
               'Jupiter Roial',
               'Jupiter celeste'},
              'Mars': {'Mars', 'Marte'},
              'Venus': {'VENERE',
               'Venere',
               'Venus',
               'Vénus',
               'la Deesse Venus',
               'la Déesse Venus'},
              'Junon': {'GIUNONE', 'Giunone', 'Juno', 'Junon'},
              'Diane': {'DIANE', 'Diana', 'Diane'},
              'Dieu': {'DEO.',
               'DIEU',
               'DIO',
               'Dieu',

### Extraction des placeName, variantes et ids

Même fonctionnement que la fonction au dessus, appliquée aux balises placeName.
On dirige les fichiers de sorties vers un second dossier nommé `placeName`.

In [20]:
def extract_texts_from_placename(folder_path, sortiePlace, NaNPlace, refPlaceInTxt):
    variantes_noms = defaultdict(set)
    namespace = {'tei': 'http://www.tei-c.org/ns/1.0'}
    non_identifies = defaultdict(set)  # Correction: set au lieu de list pour éviter les doublons
    refs_texte = defaultdict(set)
    
    total_placename = 0
    
    for root_dir, dirs, files in os.walk(folder_path):  # Parcourt récursivement tous les sous-dossiers
        print("Directory path: %s" % root_dir)
        print("Directory Names: %s" % dirs)
        print("Files Names: %s" % files)
        for file_name in files:
            if not file_name.startswith('Index') and file_name.endswith('.xml'):
                file_path = os.path.join(root_dir, file_name)
                print(f"Traitement du fichier : {file_name}")
                try:
                    # Lecture du fichier XML
                    tree = ET.parse(file_path)
                    root = tree.getroot()
                    
                    placename_count = len(root.findall('.//tei:placeName', namespaces=namespace))
                    total_placename += placename_count

                    for name in root.findall('.//tei:placeName', namespaces=namespace):
                        pers = name.text.strip() if name.text else ''
                        ref = name.attrib.get('ref', '').strip()

                        if not ref:
                            if pers:  # Seulement si le nom n'est pas vide
                                non_identifies[file_name].add(pers)
                        else:
                            if ref.startswith('#'):
                                ref = ref[1:]
                            if pers:  # Seulement si le nom n'est pas vide
                                variantes_noms[ref].add(pers)
                            refs_texte[file_name].add(ref)
                    
                except Exception as e:
                    print(f"Erreur dans le fichier {file_name}: {e}")

    sorted_var = sorted(variantes_noms.items(), key=lambda x: x[0])  # Tri alphabétique par ID
    
    with open(sortiePlace, 'w', newline='', encoding='utf-8') as csvfile:
        csvwriter = csv.writer(csvfile)
        csvwriter.writerow(['ID', 'Noms'])
        
        for ref, noms in sorted_var:
            csvwriter.writerow([ref, ','.join(sorted(noms, key=len, reverse=True))])

    with open(NaNPlace, 'w', newline='', encoding='utf-8') as NonDef:
        inconnu = csv.writer(NonDef)
        inconnu.writerow(['fileName', 'Noms'])

        for file_name, ref in non_identifies.items():
            inconnu.writerow([file_name, ','.join(sorted(ref))])

    with open(refPlaceInTxt, 'w', newline='', encoding='utf-8') as partxt:
        placeName = csv.writer(partxt)  # Correction: placeName au lieu de persName
        placeName.writerow(['fileName', 'Ref'])
        
        for file_name, ref in refs_texte.items():
            placeName.writerow([file_name, ','.join(sorted(ref))])

    print(f"Nombre total de balises placeName trouvées: {total_placename}")  # Correction: placeName

    return variantes_noms, non_identifies, refs_texte

In [18]:
sortiePlace = "placeName/variantes.csv"
NaNPlace = "placeName/Non_ID.csv"
refPlaceInTxt = "placeName/refParTexte.csv"
erreurPlace = "placeName/pbParTexte.csv"

In [21]:
extract_texts_from_placename(folder_path, sortiePlace, NaNPlace, refPlaceInTxt)

Directory path: ../../../GitHubArTerm/corpus
Directory Names: ['.git', 'Architecture', 'Peinture', 'Perspective']
Files Names: ['IndexLieux.xml', 'IndexOeuvres.xml', 'IndexPersonnes.xml', 'tei_arterm.rng']
Directory path: ../../../GitHubArTerm/corpus\.git
Directory Names: ['hooks', 'info', 'logs', 'objects', 'refs']
Files Names: ['COMMIT_EDITMSG', 'config', 'description', 'FETCH_HEAD', 'HEAD', 'index', 'ORIG_HEAD']
Directory path: ../../../GitHubArTerm/corpus\.git\hooks
Directory Names: []
Files Names: ['applypatch-msg.sample', 'commit-msg.sample', 'fsmonitor-watchman.sample', 'post-update.sample', 'pre-applypatch.sample', 'pre-commit.sample', 'pre-merge-commit.sample', 'pre-push.sample', 'pre-rebase.sample', 'pre-receive.sample', 'prepare-commit-msg.sample', 'push-to-checkout.sample', 'sendemail-validate.sample', 'update.sample']
Directory path: ../../../GitHubArTerm/corpus\.git\info
Directory Names: []
Files Names: ['exclude']
Directory path: ../../../GitHubArTerm/corpus\.git\logs
Di

(defaultdict(set,
             {'Italie': {'Italia', 'Italie'},
              'Venise': {'Etat Venitien',
               'Republique',
               'Republique de Venise',
               'Serenissima Republica',
               'Venetia',
               'Venetus',
               'Venezia',
               'Venise',
               'Venize',
               'Vénise',
               'Vinegia',
               'Vénise',
               'cette illustre vile',
               'cette vile',
               'istessa Republica'},
              'Pise': {'Pisa', 'Pise'},
              'Grece': {'Grece', 'Grecia', 'Grèce', 'Gréce'},
              'Toscane': {'Estat du grand Duc',
               'Thuscane',
               'Toscana',
               'Toscane',
               'Toscanne',
               'Tuscane',
               "l'Estat du grand\n                                    Duc"},
              'Ravenna': {'Ravenna', 'Ravenne', 'cette vile', 'même vile'},
              'Troie': {'Troia', 'Troie', 

### Création des listes à comparer à partir des fichiers crées depuis l'extraction et des fichiers index (GoogleSheets et XML)

On extrait les informations de `pers.csv` qui contient les identifiants présents sur le GoogleSheets pour les transformer en liste `id_sheets`

On extrait la liste des identifiants récupérés en même temps que les variantes d'écritures des différents noms, depuis le fichier `refParTexte` vers la liste `id_csv`. On utilise ce fichier de sortie afin de conserver l'information des fichiers dans lesquels se trouve chaque ref. 

On extrait les identifiants de l'`IndexPersonnes.xml` dans la liste `id_xml` afin de comparer à la fois les erreurs dans les textes mais également vérifier nos listes d'Index.

In [27]:
def creation_liste_persName(sheet_pers, csv_pers, xml_pers):
    # Création des listes vides
    id_sheets_pers = []
    id_csv_pers = []
    id_xml_pers = []
    
    with open(sheet_pers, 'r', encoding='utf-8') as sheet: # Lecture du fichier csv qui contient les identifiants du GoogleSheets
        reader = csv.DictReader(sheet)
        for row in reader: 
            if row['ID']:
                id_sheets_pers.append(row['ID'].strip())
                         
    with open(csv_pers, 'r', encoding='utf-8') as cfile: # Lecture du fichier csv qui contient les refs récupérées dans les fichiers xml du corpus
            reader = csv.DictReader(cfile)
            for row in reader: 
                file = row['fileName'].strip() 
                id_noms = row['Ref'].strip()
                if file and id_noms: # on utilise le fichier refParTexte pour garder l'information du classement par fichier pour retrouver les erreurs plus facilement
                    id_noms = [n.strip() for n in id_noms.split(',') if n.strip()]
                    id_csv_pers.append({'Nom du fichier': file,'ref': id_noms }) 
    
    tree = ET.parse(xml_pers)
    root = tree.getroot()
    namespace = {'tei': 'http://www.tei-c.org/ns/1.0'}
    
    for person in root.findall('.//tei:person', namespaces=namespace): # Lecture du fichier IndexPersonnes pour récupérer les ids présent dans le doc xml
        refindex = person.attrib.get('{http://www.w3.org/XML/1998/namespace}id', '').strip()
        if refindex:
            id_xml_pers.append(refindex)


    return id_sheets_pers, id_csv_pers, id_xml_pers

Fichiers nécessaires pour la création des listes.  
À ne pas modifier !

In [28]:
sheet_pers = "../balises/fichiers_noms/auteurs.csv"
csv_pers = "persName/refParTexte.csv"
xml_pers = "../../corpus/IndexPersonnes.xml"

Etape de vérification des listes.

In [29]:
id_sheets_pers, id_csv_pers, id_xml_pers = creation_liste_persName(sheet_pers, csv_pers, xml_pers)
print("Liste des IDs de sheet_file :", id_sheets_pers)
print("Liste des IDs de csv_file :", id_csv_pers)
print("Liste des IDs dans IndexPersonnes:", id_xml_pers)

Liste des IDs de sheet_file : ['ManiusValeriusMaximusCorvinusMessalla', 'GiovanniBattistaBranconioDellAquila', 'LuciusManliusCapitolinusImperiosus', 'LodovicoDiLeonardoBuonarrotiSimoni', 'ArmandJeanDuPlessisDeRichelieu', 'QuintusFabiusMaximusVerrucosus', 'IsabelleClaireEugenieDAutriche', 'AnthonieVanMontfoortBlocklandt', 'GiovanniPaoloGallucciSalodiano', 'QuintusFabiusMaximusRullianus', 'GastonDeSecondatDeMontesquieu', 'LouisIPhelypeauxDeLaVrilliere', 'GiovanniAndreaGiliodaFabriano', 'PierreLouisReichDePennautier', 'GiovanniBenedettoCastiglione', 'AntonioMariaCiocchiDelMonte', 'GiovanniBattistaDeCavalieri', 'ArmandJeanVignerotDuPlessis', 'LeopoldGuillaumeDeHabsbourg', 'FrancescoBrambillaIlGiovane', 'WolfgangGuillaumeDeNeubourg', 'AlessandroFarneseIlGiovane', 'GiovanniBattistaDellaMarca', 'AntoinePerrenotDeGranvelle', 'CarloLuigiDAragonaTagliava', 'CatherineMichelleDAutriche', 'AntonioDaSangalloIlGiovane', 'AntonioDaSangalloIlVecchio', 'GiovanniFrancescoRomanelli', 'MarcDeVulsonDeLaColo

On extrait les informations depuis le fichier `lieux.csv`

On créer des listes pour comparer les refs et ids des placeName de la même façon que pour les persName.

In [33]:
def creation_liste_placeName(sheet_place, csv_place, xml_place):
    id_sheets_place = []
    id_csv_place = []
    id_xml_place = []
    
    with open(sheet_place, 'r', encoding='utf-8') as sheet:
        reader = csv.DictReader(sheet)
        for row in reader: 
            if row['ID']:
                id_sheets_place.append(row['ID'].strip())
                         
    with open(csv_place, 'r', encoding='utf-8') as cfile:
            reader = csv.DictReader(cfile)
            for row in reader: 
                file = row['fileName'].strip() 
                id_noms = row['Ref'].strip()
                if file and id_noms: 
                    id_noms = [n.strip() for n in id_noms.split(',') if n.strip()]
                    id_csv_place.append({'Nom du fichier': file,'ref': id_noms }) 
    
    tree = ET.parse(xml_place)
    root = tree.getroot()
    namespace = {'tei': 'http://www.tei-c.org/ns/1.0'}
    
    for person in root.findall('.//tei:place', namespaces=namespace):
        refindex = person.attrib.get('{http://www.w3.org/XML/1998/namespace}id', '').strip()
        if refindex:
            id_xml_place.append(refindex)


    return id_sheets_place, id_csv_place, id_xml_place

In [34]:
sheet_place = "../balises/fichiers_noms/lieux.csv"
csv_place = "placeName/refParTexte.csv"
xml_place = "../../corpus/IndexLieux.xml"

In [35]:
id_sheets_place, id_csv_place, id_xml_place = creation_liste_placeName(sheet_place, csv_place, xml_place)
print("Liste des IDs de sheet_file :", id_sheets_place)
print("Liste des IDs de csv_file :", id_csv_place)
print("Liste des IDs dans IndexPersonnes:", id_xml_place)

Liste des IDs de sheet_file : ['SaintRemyDeProvence', 'SaintEtienneVille', 'SaintQuentinVille', 'BassanoDelGrappa', 'SantAngeloInVado', 'CittaDiCastello', 'Constantinople', 'PordenoneVille', 'ValleDiBlenio', 'Fontainebleau', 'Grottaferrata', 'AixLaChapelle', 'NeubourgDuche', 'GambassiTerme', 'MonteCavallo', 'EmpireOrient', 'ThebesEgypte', 'CastelFranco', 'FrancheComte', 'Ripatransone', 'PortoErcole', 'Schaffhouse', 'Montpellier', 'Portovenere', 'Versailles', 'Mauritanie', 'Angleterre', 'Strasbourg', 'Pessinonte', 'Alexandrie', 'Samothrace', 'Beauregard', 'Caravaggio', 'Settignano', 'Flessingue', 'Westphalie', 'Ingolstadt', 'Wurtzbourg', 'Heidelberg', 'Copenhague', 'Oudenaarde', 'Monferrato', 'Allemagne', 'Lombardie', 'Plaisance', 'Nuremberg', 'Languedoc', 'Marseille', 'Albigeois', 'Valduggia', 'Jerusalem', 'Metaponte', 'MontLiban', 'Pouzzoles', 'Augsbourg', 'Bruxelles', 'Amsterdam', 'Macedoine', 'Belvedere', 'Thessalie', 'Caprarola', 'Correggio', 'Heemskerk', 'Meulebeke', 'Francfort', 

#### Comparaisons entre les trois listes

On peut visualiser toutes les différences entre les listes. On ne sort en fichier externe que les identifiants trouvés dans les textes mais qui ne sont pas présent dans le XML. Ce fichier est `pbParTexte.csv` qui permet de voir les erreurs d'identifiants par texte. 

Ces erreurs peuvent être de plusieurs nature :
* L'identifiant n'existe pas dans le XML, oubli ou suppression dans le XML
* L'identifiant est erroné 
* L'identifiant appartient à un lieu et à été mal balisé

In [36]:
def comparaison(sheet_pers, csv_pers, xml_pers, erreurPers, sheet_place, csv_place, xml_place, erreurPlace):

    id_sheets_pers, id_csv_pers, id_xml_pers = creation_liste_persName(sheet_pers, csv_pers, xml_pers)
    id_sheets_place, id_csv_place, id_xml_place = creation_liste_placeName(sheet_place, csv_place, xml_place)
    
    id_csv_refs_pers = [] # Dans les listes id_csv on a conservé un dictionnaire pour trier les refs par texte. On récupére en une seule liste toutes les références pour les comparer et pouvoir reclasser les refs par textes plus tard
    for entry in id_csv_pers:
        id_csv_refs_pers.extend(entry['ref'])
    
    id_csv_refs_place = []
    for entry in id_csv_place:
        id_csv_refs_place.extend(entry['ref'])

   # On peut étudier toutes les différences entre toutes les listes que l'on a.

    sheetVStxt_pers = list(set(id_sheets_pers) - set(id_csv_refs_pers))
    sheetVStxt_place = list(set(id_sheets_place) - set(id_csv_refs_place))
    #print("\nIdentifiants du sheet qui ne sont pas présents dans les textes:\n", sheetVStxt_pers, "\n", sheetVStxt_place)

    sheetVSxml_pers = list(set(id_sheets_pers) - set(id_xml_pers))
    sheetVSxml_place = list(set(id_sheets_place) - set(id_xml_place))
    #print("\nIdentifiants du sheet qui ne sont pas présents dans l'index':\n", sheetVSxml_pers, "\n", sheetVSxml_place)

    txtVSxml_pers = list(set(id_csv_refs_pers) - set(id_xml_pers))
    txtVSxml_place = list(set(id_csv_refs_place) - set(id_xml_place))
    print("\nIdentifiants des textes qui ne sont pas présents dans l'index':\n", txtVSxml_pers, "\n", txtVSxml_place)

    xmlVStxt_pers = list(set(id_xml_pers) - set(id_csv_refs_pers))
    xmlVStxt_place = list(set(id_xml_place) - set(id_csv_refs_place))
    print("\nIdentifiants de l'index qui ne sont pas présents dans les textes':\n", xmlVStxt_pers, "\n", xmlVStxt_place)

    txtVSsheet_pers = list(set(id_csv_refs_pers) - set(id_sheets_pers))
    txtVSsheet_place = list(set(id_csv_refs_place) - set(id_sheets_place))
    #print("\nIdentifiants des textes qui ne sont pas présents dans le sheet:\n", txtVSsheet_pers, "\n", txtVSsheet_place)

    xmlVSsheet_pers = list(set(id_xml_pers) - set(id_sheets_pers))
    xmlVSsheet_place = list(set(id_xml_place) - set(id_sheets_place))
    #print("\nIdentifiants de l'index qui ne sont pas présents dans le sheet:\n", xmlVSsheet_pers, "\n", xmlVSsheet_place)

    # On écrit deux fichiers csv avec les identifiants qui sont dans les textes mais pas dans l'IndexPersonnes ou Lieux.
    with open(erreurPers, 'w', newline='', encoding='utf-8') as ErreurPers:
            pb = csv.writer(ErreurPers)
            pb.writerow(['fileName', 'Ref'])
            
            for entry in id_csv_pers:
                file_name = entry['Nom du fichier']
                refs_diff = [ref for ref in entry['ref'] if ref in txtVSxml_pers]
                if refs_diff:
                    pb.writerow([file_name, ','.join(sorted(refs_diff))])

    with open(erreurPlace, 'w', newline='', encoding='utf-8') as ErreurPlace:
            pb = csv.writer(ErreurPlace)
            pb.writerow(['fileName', 'Ref'])
            
            for entry in id_csv_place:
                file_name = entry['Nom du fichier']
                refs_diff = [ref for ref in entry['ref'] if ref in txtVSxml_place]
                if refs_diff:
                    pb.writerow([file_name, ','.join(sorted(refs_diff))])
                
    return sheetVStxt_pers, sheetVSxml_pers, txtVSxml_pers, txtVSsheet_pers, xmlVSsheet_pers, xmlVStxt_pers, sheetVStxt_place, sheetVSxml_place, txtVSxml_place, txtVSsheet_place, xmlVSsheet_place, xmlVStxt_place

### Executer le programme.

Toutes les variables attendues par la fonction `comparaison()` ont été définies plus tôt pour l'utilisation des autres fonctions. Il suffit donc d'exécuter cette dernière cellule pour lancer la comparaison des différentes listes.

In [37]:
comparaison(sheet_pers, csv_pers, xml_pers, erreurPers, sheet_place, csv_place, xml_place, erreurPlace)


Identifiants des textes qui ne sont pas présents dans l'index':
 ['Peirasos', 'SigismondoFoliano', 'JeanBarbe', 'Boccace', 'Jason', 'Agathocle', 'Epimenide', 'AlphonseIIEste', 'Anubis', 'CharlesDeLEcluse', 'AgostinoTassi', 'PietroCapocci', 'TatienSyrien', 'CristoforoCasolani', 'GiovanniBattistaNaldini', 'GiovanniBattistaFonteo', 'Talos', 'Properce', 'RobertDeLenoncourt', 'Hyperbius', 'SantAmbroise', 'BaldassarreCroce', 'CaiusMaecenas', 'AppiusClaudiusCaecus', 'SaintRemi', 'Aristee', 'Sesostris', 'PaoloSaetoniGrimaldi', 'SaintAntoineDePadoue', 'SebastianoVannini', 'Typhon', 'Calcostenes', 'Nemesis', 'Gorsanus', 'NicoloMartinelli', 'Gaudentio', 'JeanMace', 'Iphicles', 'BernardoCesi', 'Saserna', 'EtienneDeByzance', 'DonRodrigoDiToledo', 'AdrienIII', 'SaintApollinaire', 'GiovanniBattistaDellaMarca', 'Iolaos', 'CambyseII', 'HydreDeLerne', 'JuliusFirmicusMaternus', 'Demophile', 'Morliere', 'GianVittorioRossi', 'JacquesBordier', 'FrancescoTraballesi', 'Phlegias', 'GottardoDaPonte', 'Ostie', 

(['Nestor',
  'MartiaVarrone',
  'ValentinumBoltzen',
  'GiacomoMazzocchi',
  'FrancescoDomenicoBisagno',
  'FilippoValori',
  'GinevraBenci',
  'AscanioPersio',
  'JostAmman',
  'IppolitoDonesmondi',
  'Charites',
  'FredericArmandDeSchomberg',
  'DenysDeColophon',
  'AntigoneleBorgne',
  'CleodoroCalchi',
  'LioneldEste',
  'GiovanniAmbrogioMazenta',
  'BenedettoAlberti',
  'LaurentiusAlbertus',
  'Narcisse',
  'DonatoAcciaiuoli',
  'PhiloclesdEgypte',
  'GiovanniAndreaGiliodaFabriano',
  'Alberti',
  'GiovanniPaoloGallucciSalodiano',
  'MelchisedecThevenot',
  'PierreBourdelot',
  'Borromeo',
  'IsottadegliAtti',
  'Cassandre',
  'AldoManuzio',
  'AlessandroLamo',
  'GiovanniAlberti2',
  'GaleazzoArconati',
  'EtienneGuillery',
  'Zephyr',
  'CarloAlberti',
  'CamilloAlbizzi',
  'AndreaSalaino',
  'PieIII',
  'Achemenide',
  'HenryPeachamjunior',
  'Protagoras',
  'Phanostrate',
  'Perseus',
  'MichelinoDaBesozzo',
  'GentileBorri',
  'SaintPotit',
  'Tasse',
  'OrazioMelzi',
  'Ala

## Comparaison entre les index et les refs utilisées

On sépare les différentes erreurs :
* les personnes annotées avec des balises placeName
* les lieux annotés avec des balises persName
* les identifiants qui ne figurent dans aucuns Index.

In [38]:
def cross_comparaison(sheet_pers, csv_pers, xml_pers, sheet_place, csv_place, xml_place, erreurs):

    id_sheets_pers, id_csv_pers, id_xml_pers = creation_liste_persName(sheet_pers, csv_pers, xml_pers)
    id_sheets_place, id_csv_place, id_xml_place = creation_liste_placeName(sheet_place, csv_place, xml_place)
    
    id_csv_refs_pers = []
    for entry in id_csv_pers:
        id_csv_refs_pers.extend(entry['ref'])
    
    id_csv_refs_place = []
    for entry in id_csv_place:
        id_csv_refs_place.extend(entry['ref'])

    # Liste des identifiants des listes tirées des texte qui ne sont pas dans les indexs
    ID_inexistant = [x for x in id_csv_refs_pers + id_csv_refs_place if x not in id_xml_pers and x not in id_xml_place]
    # Liste des identifiants dans des balises persName qui sont dans l'IndexLieux (balisé avec persName au lieu de placeName)
    place_dans_PersName = [x for x in id_csv_refs_pers if x not in id_xml_pers and x in id_xml_place]
    # Liste des identifiants dans des balises placeName qui sont dans l'IndexPersonnes (balisé avec placeName au lieu de persName)
    pers_dans_PlaceName = [x for x in id_csv_refs_place if x not in id_xml_place and x in id_xml_pers]
    
    # On écrit un fichier csv avec les trois listes créées au dessus pour afficher les résultats plus clairement avec un tri par fichier
    with open(erreurs, 'w', newline='', encoding='utf-8') as Erreur:
            pb = csv.writer(Erreur)
            pb.writerow(["Identifiants inexistants\n"])
            pb.writerow(['fileName', 'Ref'])
            
            for entry in id_csv_place + id_csv_pers:
                file_name = entry['Nom du fichier']
                refs_diff = [ref for ref in entry['ref'] if ref in ID_inexistant]
                if refs_diff:
                    pb.writerow([file_name, ','.join(sorted(refs_diff))])
            
            pb.writerow(["\nLieux dans des balises persName\n"])
            for entry in id_csv_pers:
                file_name = entry['Nom du fichier']
                refs_diff = [ref for ref in entry['ref'] if ref in place_dans_PersName]
                if refs_diff:
                    pb.writerow([file_name, ','.join(sorted(refs_diff))])

            pb.writerow(["\nPersonnes dans des balises placeName\n"])
            for entry in id_csv_place:
                file_name = entry['Nom du fichier']
                refs_diff = [ref for ref in entry['ref'] if ref in pers_dans_PlaceName]
                if refs_diff:
                    pb.writerow([file_name, ','.join(sorted(refs_diff))])

    print("\nIdentifiants trouvé dans des balises persName qui sont dans IndexLieux: \n", place_dans_PersName)
    print("\nIdentifiants trouvé dans des balises place qui sont dans IndexPersonnes: \n", pers_dans_PlaceName)
    print("\nLes identifiants n'existent dans aucun index XML:\n", ID_inexistant)


    return ID_inexistant, place_dans_PersName, pers_dans_PlaceName

Un fichier de sortie unique qui sépare les différentes erreurs tout en indiquant dans quel fichier XML elles se trouvent.

In [39]:
erreurs = "erreurs.csv"

## Utilisation de la fonction pour cross check les listes 

In [40]:
cross_comparaison(sheet_pers, csv_pers, xml_pers, sheet_place, csv_place, xml_place, erreurs)


Identifiants trouvé dans des balises persName qui sont dans IndexLieux: 
 ['Ostie', 'Tyr']

Identifiants trouvé dans des balises place qui sont dans IndexPersonnes: 
 ['Antiochos', 'AuluGelle']

Les identifiants n'existent dans aucun index XML:
 ['JacopoDeBarberi', 'AlphonseIIEste', 'Arcesilas', 'BernardoCesi', 'Calcostenes', 'Canachus', 'CharlesDeLEcluse', 'Demophile', 'Dubie', 'EtienneDeByzance', 'GeorgiusAgricola', 'Gorsanus', 'IsaacGribelin', 'JacquesBordier', 'JeanMace', 'LeonardoGarzoni', 'LouisHance', 'Lysistratos', 'MathurinJousse', 'Morliere', 'Phyromachos', 'PierreChartier', 'RobertVauquer', 'Talos', 'AdrienIII', 'Agathocle', 'Anubis', 'Apis', 'AppiusClaudiusCaecus', 'Aristee', 'AristobuleDeCassandreia', 'Boccace', 'CaiusMaecenas', 'CambyseI', 'CambyseII', 'Demosthene', 'Denys', 'Epimenide', 'Eratosthene', 'Eunostos', 'Euryalus', 'Frontin', 'Hippodamos', 'HydreDeLerne', 'Hyperbius', 'Iolaos', 'Iphicles', 'Iphicrate', 'Jason', 'JuliusFirmicusMaternus', 'Latone', 'LuciusTaruti

(['JacopoDeBarberi',
  'AlphonseIIEste',
  'Arcesilas',
  'BernardoCesi',
  'Calcostenes',
  'Canachus',
  'CharlesDeLEcluse',
  'Demophile',
  'Dubie',
  'EtienneDeByzance',
  'GeorgiusAgricola',
  'Gorsanus',
  'IsaacGribelin',
  'JacquesBordier',
  'JeanMace',
  'LeonardoGarzoni',
  'LouisHance',
  'Lysistratos',
  'MathurinJousse',
  'Morliere',
  'Phyromachos',
  'PierreChartier',
  'RobertVauquer',
  'Talos',
  'AdrienIII',
  'Agathocle',
  'Anubis',
  'Apis',
  'AppiusClaudiusCaecus',
  'Aristee',
  'AristobuleDeCassandreia',
  'Boccace',
  'CaiusMaecenas',
  'CambyseI',
  'CambyseII',
  'Demosthene',
  'Denys',
  'Epimenide',
  'Eratosthene',
  'Eunostos',
  'Euryalus',
  'Frontin',
  'Hippodamos',
  'HydreDeLerne',
  'Hyperbius',
  'Iolaos',
  'Iphicles',
  'Iphicrate',
  'Jason',
  'JuliusFirmicusMaternus',
  'Latone',
  'LuciusTarutiusFirmanus',
  'Martial',
  'Nemesis',
  'Ops',
  'Parmenion',
  'Peirasos',
  'Phlegias',
  'Pisistrate',
  'Properce',
  'PseudoNechepsos',
  